# R02 Probe Evaluation - retrieval-first graph shape

**Author**: Knowledge Graph Foundry autonomous build

Measures the pre-registered R02 hypotheses on the 28-probe CPAP set (`tests/probes/cpap-probe-set.yml`). Phase A ablates on the R01 graph: A0 baseline retrieval (propositions off), A1 propositions on (H11), A2 after similarity-edge densification (H13). Phase B (separate run) measures the H10/H12 rebuild graph. Scoring is deterministic: evidence recall by normalized substring match, answer-value match on numeric/value tokens, refusal regex for abstention.

In [1]:
# imports
import json  # results persistence
import os  # env overrides
import re  # scoring
from pathlib import Path  # paths

import yaml  # probe set

from knowledge_graph_foundry import Foundry, load_settings  # engine under test

2026-07-06 13:24:08.294 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


In [2]:
# configuration
PROBES_PATH = Path('../tests/probes/cpap-probe-set.yml')  # gold probe set
REPORTS = Path('../reports')  # results output
PHASE = os.environ.get('KGF_EVAL_PHASE', 'A')  # A = R01 graph ablations, B = rebuild graph
NEO4J_URI = os.environ.get('KGF_EVAL_URI')  # optional graph override (phase B)
print(f'phase={PHASE} uri={NEO4J_URI or "(default env)"}')

phase=A uri=(default env)


In [3]:
# probe set
probes = yaml.safe_load(PROBES_PATH.read_text())
print(len(probes), 'probes:', {c: sum(1 for p in probes if p['category'] == c)
                               for c in ('single_fact', 'comparison', 'multi_hop', 'unanswerable')})

28 probes: {'single_fact': 10, 'comparison': 8, 'multi_hop': 6, 'unanswerable': 4}


In [4]:
# scoring - deterministic, reproducible
REFUSAL = re.compile(
    r'no information|not (?:available|stated|specified|mentioned|provided)|'
    r'lacks|does not (?:contain|include|specify|state|provide|mention)|'
    r'cannot answer|unable to|unanswerable|no (?:data|details|answer)', re.I)

def _norm(s):
    return re.sub(r'\s+', ' ', s.casefold())

def evidence_recall(gold_evidence, context):
    if not gold_evidence:
        return None
    ctx = _norm(context)
    return sum(1 for g in gold_evidence if _norm(g) in ctx) / len(gold_evidence)

def value_tokens(gold_answer):
    return [t for t in re.findall(r'[\w.\-/]*\d[\w.\-/]*', gold_answer)]

def answer_correct(probe, answer):
    ans = _norm(answer)
    if probe['category'] == 'unanswerable':
        return bool(REFUSAL.search(answer))
    tokens = value_tokens(probe['gold_answer'])
    if tokens:
        hit = sum(1 for t in tokens if _norm(t) in ans)
        return hit >= max(1, len(tokens) // 2 + (len(tokens) % 2))
    return _norm(probe['gold_answer']) in ans

def false_refusal(probe, answer):
    return probe['category'] != 'unanswerable' and bool(REFUSAL.search(answer))

In [5]:
# eval runner - one pass over all probes for a given foundry
def run_eval(foundry, label):
    rows = []
    for p in probes:
        context_lines, _ = foundry._retrieve_local(p['question'])
        context = '\n'.join(context_lines)
        result = foundry.query(p['question'])
        answer = result['answer']
        rows.append({
            'id': p['id'], 'category': p['category'],
            'evidence_recall': evidence_recall(p['gold_evidence'], context),
            'correct': answer_correct(p, answer),
            'false_refusal': false_refusal(p, answer),
            'context_chars': len(context),
            'answer': answer,
        })
        print(f"{p['id']} {p['category']:<13} recall="
              f"{rows[-1]['evidence_recall'] if rows[-1]['evidence_recall'] is not None else '-'} "
              f"correct={rows[-1]['correct']}")
    return {'label': label, 'rows': rows}

def summarize(result):
    rows = result['rows']
    answerable = [r for r in rows if r['category'] != 'unanswerable']
    unans = [r for r in rows if r['category'] == 'unanswerable']
    recalls = [r['evidence_recall'] for r in answerable if r['evidence_recall'] is not None]
    by_cat = {}
    for cat in ('single_fact', 'comparison', 'multi_hop'):
        cat_rows = [r for r in rows if r['category'] == cat]
        by_cat[cat] = sum(r['correct'] for r in cat_rows) / len(cat_rows) if cat_rows else None
    return {
        'label': result['label'],
        'evidence_recall': sum(recalls) / len(recalls) if recalls else 0,
        'answer_accuracy': sum(r['correct'] for r in answerable) / len(answerable),
        'accuracy_by_category': by_cat,
        'correct_refusal': sum(r['correct'] for r in unans) / len(unans) if unans else None,
        'false_refusal': sum(r['false_refusal'] for r in answerable) / len(answerable),
        'avg_context_chars': int(sum(r['context_chars'] for r in rows) / len(rows)),
    }

## Phase A - ablations on the R01 graph

A0: propositions disabled (R01 retrieval baseline). A1: propositions enabled (H11 lever). A2: after the similarity-edge pass (H13 lever; graph state change, run once).

In [6]:
results = {}
settings = load_settings()
if NEO4J_URI:
    settings.neo4j.uri = NEO4J_URI

settings.graphrag.propositions_enabled = False
with Foundry(settings.model_copy(deep=True)) as f:
    results['A0_baseline'] = run_eval(f, 'A0_baseline')
print(summarize(results['A0_baseline']))

2026-07-06 13:24:08.715 | WARNING  | knowledge_graph_foundry.settings:load_settings:143 - config file config.yml not found, using defaults


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 13:25:33.028 | INFO     | knowledge_graph_foundry.extraction.embeddings:_embed_bedrock:91 - Embeddings: 1/1
2026-07-06 13:25:33.030 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:199 - Generated embeddings for 1/1 entities via bedrock (0 cache hits)
2026-07-06 13:25:33.040 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 13:25:33.067 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:25:33.077 | WARNING  | knowledge_

P28 unanswerable  recall=- correct=True
{'label': 'A0_baseline', 'evidence_recall': 0.3958333333333333, 'answer_accuracy': 0.7083333333333334, 'accuracy_by_category': {'single_fact': 0.8, 'comparison': 0.875, 'multi_hop': 0.3333333333333333}, 'correct_refusal': 1.0, 'false_refusal': 0.3333333333333333, 'avg_context_chars': 3253}


In [7]:
settings.graphrag.propositions_enabled = True
with Foundry(settings.model_copy(deep=True)) as f:
    results['A1_propositions'] = run_eval(f, 'A1_propositions')
print(summarize(results['A1_propositions']))

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


2026-07-06 13:25:35.122 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:25:35.186 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 13:25:35.208 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:25:35.225 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed 

...

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P27 unanswerable  recall=- correct=True
2026-07-06 13:26:48.807 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:26:48.821 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 13:26:48.839 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:26:48.853 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedur

P28 unanswerable  recall=- correct=True
{'label': 'A1_propositions', 'evidence_recall': 0.4375, 'answer_accuracy': 0.75, 'accuracy_by_category': {'single_fact': 0.9, 'comparison': 0.875, 'multi_hop': 0.3333333333333333}, 'correct_refusal': 1.0, 'false_refusal': 0.25, 'avg_context_chars': 3876}


In [8]:
# H13: densify (idempotent), then re-run with propositions on
from knowledge_graph_foundry.graph.densify import add_similarity_edges
with Foundry(settings.model_copy(deep=True)) as f:
    created = add_similarity_edges(
        f.driver, f.settings.graphrag.vector_index_name,
        threshold=f.settings.graphrag.similarity_threshold,
        top_k=f.settings.graphrag.similarity_top_k)
    print('similarity edges created:', created)
    results['A2_densified'] = run_eval(f, 'A2_densified')
print(summarize(results['A2_densified']))

Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `SIMILAR_TO` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=13, offset=12>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 12, 'line': 1, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH ()-[r:SIMILAR_TO]->() RETURN count(r) AS n'


2026-07-06 13:26:52.789 | INFO     | knowledge_graph_foundry.graph.densify:add_similarity_edges:52 - similarity edges: 500/4601 entities


2026-07-06 13:26:54.058 | INFO     | knowledge_graph_foundry.graph.densify:add_similarity_edges:52 - similarity edges: 1000/4601 entities


2026-07-06 13:26:55.358 | INFO     | knowledge_graph_foundry.graph.densify:add_similarity_edges:52 - similarity edges: 1500/4601 entities


2026-07-06 13:26:56.592 | INFO     | knowledge_graph_foundry.graph.densify:add_similarity_edges:52 - similarity edges: 2000/4601 entities


...

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL gds.graph.drop($name, false)'


P27 unanswerable  recall=- correct=True
2026-07-06 13:28:18.827 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:28:18.840 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedure.ProcedureCallFailed} {message: Failed to invoke procedure `gds.graph.project`: Caused by: java.lang.IllegalArgumentException: Invalid node projection, one or more labels not found: 'Entity, Chunk'} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}
2026-07-06 13:28:18.859 | INFO     | knowledge_graph_foundry.extraction.embeddings:generate_embeddings:188 - Embeddings: 1/1 from cache
2026-07-06 13:28:18.872 | WARNING  | knowledge_graph_foundry.graph.graphrag:ppr_query:187 - PPR query fell back (graph issue): {neo4j_code: Neo.ClientError.Procedur

P28 unanswerable  recall=- correct=True
{'label': 'A2_densified', 'evidence_recall': 0.4375, 'answer_accuracy': 0.75, 'accuracy_by_category': {'single_fact': 0.9, 'comparison': 0.875, 'multi_hop': 0.3333333333333333}, 'correct_refusal': 1.0, 'false_refusal': 0.25, 'avg_context_chars': 3877}


In [9]:
# summary table + persist
import datetime
summaries = [summarize(r) for r in results.values()]
for s in summaries:
    print(f"{s['label']:<16} evrecall={s['evidence_recall']:.3f} "
          f"acc={s['answer_accuracy']:.3f} refusal_ok={s['correct_refusal']} "
          f"false_refusal={s['false_refusal']:.3f} ctx={s['avg_context_chars']}")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')
out = REPORTS / f'probe-eval-phase{PHASE}-{stamp}.json'
out.write_text(json.dumps({'summaries': summaries, 'results': results}, indent=2, default=str))
print('saved', out)

A0_baseline      evrecall=0.396 acc=0.708 refusal_ok=1.0 false_refusal=0.333 ctx=3253
A1_propositions  evrecall=0.438 acc=0.750 refusal_ok=1.0 false_refusal=0.250 ctx=3876
A2_densified     evrecall=0.438 acc=0.750 refusal_ok=1.0 false_refusal=0.250 ctx=3877
saved ../reports/probe-eval-phaseA-20260706-112820.json
